# Woody Hayes Dr. Construction Analysis

This notebook is intended to analyze if the lane restrictions on Woody Hayes Dr. starting on March 26, 2026 have impacted metrics for the medical center shuttle.


---

### Background:

On March 26th, 2026, Woody Hayes Dr. between Kenny Rd. and John H Herrick Dr. was reduced from 2 eastbound lanes and 2 westbound lanes with a center median, to 1 lane each direction. This is due to construction occuring on the adjacent lot intended to end in mid June 2026. The following questions intend to be answered by this analysis:

1) Has this lane restriction caused a change in travel time for the med center express shuttle?
2) Are buses 'bunching' due to traffic in the area?
    - This will be defined by close arrival times of multiple busses, or rather how spread out the busses appear to be in terms of arrival time.

### Relevant Information:

Busstate data for this analysis was pulled on 4/29/2026 and is stored at the following file path:

`../WMC_Dashboard/analysis_data/woody_hayes/`

This includes busstate data from February - April 2026

---

### Setup:

In [1]:
# Lane restriction was implemented at some point on 3/36/2026
# Going to pull in data and split around this date for before and after analysis

# import libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import os
import importlib

In [2]:
# go up one level from notebook folder
sys.path.append(os.path.abspath(".."))

from modules import med_center_dashboard as mcd
importlib.reload(mcd)

repo_dir = "K:/AP/TTM/Data/WMC_Dashboard/analysis_data/woody_hayes"

# month and date dict
timeframe_dict = {
    # 'JAN': 2026,
    'FEB': 2026,
    'MAR': 2026,
    'APR': 2026
}

# loop through and concat to once single df
df = pd.DataFrame()
for month, year in timeframe_dict.items():
    df = pd.concat([df, mcd.process_mc_busstate(year=year, month=month, current_dir=repo_dir)], ignore_index=True)

# This should only have to be run once per session. ~9 minutes to run

In [14]:
# split into before and after timeframes with equal number of days on either side
# not inclusive of 3/26/2026 as the lane restriction could have been implemented at any point during that day
before_df = df[(df['DATE'] >= '2026-02-25') & (df['DATE'] < '2026-03-26')] # 21 business days 2/25 - 3/25
after_df = df[(df['DATE'] > '2026-03-26') & (df['DATE'] <= '2026-04-24')] # 21 business days 3/27 - 4/24

In [15]:
print(f'Before lane restriction: {len(before_df)} records')
print(f'After lane restriction: {len(after_df)} records') # even out records

Before lane restriction: 34574 records
After lane restriction: 34779 records



---

### Has the lane restriction led to a change in travel times?

Specifically in the ~3-5pm hour

In [16]:
# import travel_time module as initial check
from modules import travel_time 

before_travel_time = travel_time.create_travel_time(before_df)
after_travel_time = travel_time.create_travel_time(after_df)

print("Before lane restriction travel time stats:")
display(before_travel_time)

print("After lane restriction travel time stats:")
display(after_travel_time)

Before lane restriction travel time stats:


,TIME,LOOPS,CARMACK 2 - UH/DOAN,UH/DOAN - CARMACK 2
0,5:30-7a,803,7.9,6.5
1,7-10a,1025,10.8,7.0
2,10a-4p,1264,10.3,6.7
3,4-7p,1027,10.2,6.5
4,7p-12a,1163,10.0,5.8
5,12-5a,403,9.5,5.4


After lane restriction travel time stats:


,TIME,LOOPS,CARMACK 2 - UH/DOAN,UH/DOAN - CARMACK 2
0,5:30-7a,808,7.9,6.4
1,7-10a,991,10.8,7.0
2,10a-4p,1280,10.2,7.0
3,4-7p,1022,10.3,7.9
4,7p-12a,1231,10.0,5.9
5,12-5a,401,9.7,5.5


Initial look given my existing module. Travel time appear to be about the same with a decent increase in the travel time specifically in the 4pm-7pm outbound timeframe. The 10a-4p timeframe also has a slight increase in time.


---

Isolating the outbound travel, timedelta of Doan Hall departure to Carmack 2 arrival

**Statistical Testing**

`H0`: mean travel time before lane closure = mean travel time after lane closure

`HA`: Mean travel time after > mean travel time before
- This is one sided since we care specifically about increases here.

In [ ]:
# df already has hour buckets

before_df.head()

# specifically want the travel time from departure of Doan Hall to arrival at Carmack 2
# before_outbound_df = before_df

,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE,BOARDING_DIRECTIONS,TIME
1227,1305,2026-02-26,1225,403,0.0,0.0,1,1503.0,MC,1900-01-01 14:19:07,1900-01-01 14:19:50,0 days 00:00:43,14,19,IB,2026-02-26 14:19:07
1228,1305,2026-02-26,1226,404,2.0,0.0,3,1503.0,MC,1900-01-01 14:20:35,1900-01-01 14:21:06,0 days 00:00:31,14,20,IB,2026-02-26 14:20:35
1229,1305,2026-02-26,1227,94,0.0,0.0,3,1503.0,MC,1900-01-01 14:22:06,1900-01-01 14:22:22,0 days 00:00:16,14,22,IB,2026-02-26 14:22:06
1230,1305,2026-02-26,1228,95,1.0,1.0,3,1503.0,MC,1900-01-01 14:22:59,1900-01-01 14:23:34,0 days 00:00:35,14,22,IB,2026-02-26 14:22:59
1231,1305,2026-02-26,1229,401,4.0,2.0,5,1503.0,MC,1900-01-01 14:29:51,1900-01-01 14:30:49,0 days 00:00:58,14,29,OB,2026-02-26 14:29:51
